# Loading dataset

In [88]:
import pandas as pd

In [89]:
df=pd.read_csv('../datasets/olist_master_dataset.csv')
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_state,total_payment_value,payment_installments,payments_method_count,primary_payment_type,review_score,review_comment_title,review_comment_message,product_category,is_delivered
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,...,SP,38.71,3.0,2.0,credit_card,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",housewares,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,...,BA,141.46,1.0,1.0,boleto,4.0,Muito boa a loja,Muito bom o produto.,perfumery,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,...,GO,179.12,3.0,1.0,credit_card,5.0,NaN,NaN,auto,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1,d0b61bfb1de832b15ba9d266ca96e5b0,...,RN,72.20,1.0,1.0,credit_card,5.0,NaN,O produto foi exatamente o que eu esperava e e...,pet_shop,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,65266b2da20d04dbe00c5c2d3bb7859e,...,SP,28.62,1.0,1.0,credit_card,5.0,NaN,NaN,stationery,1


In [90]:
df.shape

(113314, 27)

In [91]:
reviews=df[['review_score','review_comment_title','review_comment_message']]

In [92]:
reviews.isna().sum()

review_score                  0
review_comment_title      99880
review_comment_message    65672
dtype: int64

# Concating the title and message of the reviews

In [93]:
reviews['review_comment_title']=reviews['review_comment_title'].fillna("")
reviews['review_comment_message']=reviews['review_comment_message'].fillna("")

C:\Users\vighn\AppData\Local\Temp\ipykernel_9492\599396796.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews['review_comment_title']=reviews['review_comment_title'].fillna("")
C:\Users\vighn\AppData\Local\Temp\ipykernel_9492\599396796.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews['review_comment_message']=reviews['review_comment_message'].fillna("")


In [94]:
reviews['review']=reviews['review_comment_title']+" "+reviews['review_comment_message']

C:\Users\vighn\AppData\Local\Temp\ipykernel_9492\3334092743.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews['review']=reviews['review_comment_title']+" "+reviews['review_comment_message']


In [95]:
reviews[reviews['review']==" "]

,review_score,review_comment_title,review_comment_message,review
2,5.0,,,
4,5.0,,,
5,4.0,,,
7,5.0,,,
8,1.0,,,
...,...,...,...,...
113305,5.0,,,
113306,5.0,,,
113307,5.0,,,
113308,5.0,,,


In [96]:
reviews = reviews[reviews['review'].str.strip() != ""]

In [97]:
reviews=reviews[['review_score','review']]

In [98]:
reviews

,review_score,review
0,4.0,"Não testei o produto ainda, mas ele veio corr..."
1,4.0,Muito boa a loja Muito bom o produto.
3,5.0,O produto foi exatamente o que eu esperava e ...
6,2.0,fiquei triste por n ter me atendido.
10,1.0,Aguardando retorno da loja
...,...,...
113303,1.0,"Ele não é um mini cajon, é um shaker, ou seja..."
113309,4.0,So uma peça que veio rachado mas tudo bem rs
113310,5.0,Foi entregue antes do prazo.
113311,2.0,Foi entregue somente 1. Quero saber do outro ...


# Preprocessing the text

In [103]:
import re
from nltk.corpus import stopwords
from nltk.stem.snowball import PortugueseStemmer
stemmer=PortugueseStemmer()
stop_words=set(stopwords.words('portuguese'))
def preprocess(text):
    text=str(text)
    text=text.lower()
    text=re.sub(r'[^A-Za-zÀ-ÿ\s]', ' ', text)
    text=re.sub(r'\s+', ' ', text).strip()
    tokens=text.split()
    tokens=[word for word in tokens if word not in stop_words]
    tokens=[stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

In [104]:
reviews['review']=reviews['review'].apply(preprocess)

In [107]:
reviews['review_score'].unique()

array([4., 5., 2., 1., 3.])

# Classifing the score as Negative and Positive

In [114]:
def classify_score(num):
    if num in [1,2]:
        return "Negative"
    elif num in [4,5]:
        return "Positive"
    else:
        return  None

In [115]:
reviews['review_verdict']=reviews['review_score'].apply(classify_score)

In [120]:
reviews.dropna(inplace=True)

In [122]:
reviews['review_verdict'].value_counts()

review_verdict
Positive    31143
Negative    13988
Name: count, dtype: int64

# Training the Model

In [134]:
from sklearn.model_selection import train_test_split as tts
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix,classification_report

In [135]:
final_reviews=reviews[['review','review_verdict']]

In [136]:
X=final_reviews['review'].astype(str)
y=final_reviews['review_verdict']

In [137]:
X_train,X_test,y_train,y_test=tts(X,y,train_size=0.85,random_state=42,stratify=y)

In [138]:
vectorizer=TfidfVectorizer(max_features=5000,ngram_range=(1,2),min_df=5)

In [139]:
X_train_vec=vectorizer.fit_transform(X_train)
X_test_vec=vectorizer.transform(X_test)

In [140]:
model=MultinomialNB()
model.fit(X_train_vec,y_train)

MultinomialNB()

In [141]:
y_pred=model.predict(X_test_vec)
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix (rows: true, cols: pred):")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    Negative     0.8596    0.8899    0.8745      2098
    Positive     0.9498    0.9347    0.9422      4672

    accuracy                         0.9208      6770
   macro avg     0.9047    0.9123    0.9083      6770
weighted avg     0.9218    0.9208    0.9212      6770

Confusion matrix (rows: true, cols: pred):
[[1867  231]
 [ 305 4367]]


# Saving The model

In [142]:
from joblib import dump
dump(vectorizer,'../backend/models/sentiment_vectorizer.joblib')
dump(model,'../backend/models/sentiment_model.joblib')

['../backend/models/sentiment_model.joblib']